In [2]:
import sqlite3
import pandas as pd
import ast
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.proportion import proportion_confint
import numpy as np

con=sqlite3.connect(r"C:\Users\muhmm\ipl_proj\Data\Raw\ipl.db")
def q(sql):
    return pd.read_sql_query(sql,con)

In [3]:
stats.binomtest(613, 1187, 0.5).pvalue

np.float64(0.2700389712415702)

In [4]:
#Does winning the toss help you win the match?

q('''select count(*) as n,
        sum(case when toss_winner=match_winner then 1 else 0 end) as k
from matches
where result = 'win';''')

,n,k
0,1187,613


In [5]:
p = stats.binomtest(613, 1187, 0.5).pvalue
ci=proportion_confint(613, 1187, alpha=0.05, method='wilson')
print(p)
print(ci)

0.2700389712415702
(0.48799203833275506, 0.5447579131499739)


In [6]:
#Does the decision taken after the toss matter?
#h0- Captains who choose to bat and captains who 
# choose to field win equally often. The gap we saw is luck.
#h1-The two choices do not win equally often.

q("""SELECT toss_decision,
          SUM(CASE WHEN toss_winner = match_winner THEN 1 ELSE 0 END) AS won,
          SUM(CASE WHEN toss_winner <> match_winner THEN 1 ELSE 0 END) AS lost
   FROM matches_clean
   WHERE result = 'win'
   GROUP BY toss_decision;""")

,toss_decision,won,lost
0,bat,164,194
1,field,377,311


In [7]:
table =[[184,218],
        [429,356]]
chi2,p,dof,expected = stats.chi2_contingency(table)
print(round(chi2,2), round(p,4))

8.04 0.0046


In [8]:
#Does the chasing team really have an advantage?
#h0-Both sides win half the time. The 54.51% we saw is luck.
#h1-A chase does not succeed half the time.
q('''select count(*) as n,
        sum(chase_won) as k
    from v_match_totals;''')

,n,k
0,1046,571


In [9]:
q("""select case when first_innings_runs < 140 then 'under 140'
            when first_innings_runs <160 then '140-159'
            when first_innings_runs < 180 then '160-179'
            when first_innings_runs < 200 then '180-199'
            else '200+' end as band,
        sum(chase_won) as won,
        sum(1-chase_won) as lost
from v_match_totals
group by band;""")

,band,won,lost
0,140-159,148,61
1,160-179,149,132
2,180-199,73,119
3,200+,36,131
4,under 140,165,32


In [10]:
table=[[171,70],[166,146],[83,134],[46,152],[181,38]]
chi2,p,dof,expected =stats.chi2_contingency(table)
print(round(chi2,2))
print(p)

197.68
1.1854224475712924e-41
